In [1]:
 # Initialize Spark session
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hash
from pyspark.sql.functions import count, when

In [2]:
# Import SparkSession (main entry point to create Spark applications)
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .master("local[2]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.default.parallelism", "2") \
    .getOrCreate()
print("Spark Session Created Successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/07 21:11:46 WARN Utils: Your hostname, Sais-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.223.134.171 instead (on interface en0)
26/06/07 21:11:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 21:11:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


In [3]:
# Load the CSV dataset into a Spark DataFrame with header and automatic schema detection
df = spark.read.csv("/Users/saiteja/Documents/Assignment/dataset.csv", header=True, inferSchema=True)

In [4]:
# Display the dataset schema (column names and data types)
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- book_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- date_updated: string (nullable = true)
 |-- read_at: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- n_votes: string (nullable = true)
 |-- n_comments: string (nullable = true)



In [5]:
# Show the first 20 rows of the dataset
df.show(20)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+
|             user_id|             book_id|           review_id|              rating|         review_text|          date_added|        date_updated|             read_at|          started_at|             n_votes|n_comments|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+
|8842281e1d1347389...|            24375664|5cd416f3efc3f944f...|                   5|Mind blowingly co...|                NULL|                NULL|                NULL|                NULL|                NULL|      NULL|
| The undulations ...| but I wouldn't h...| and I wouldn't h...|                NULL|                NULL|  

In [6]:
# Import os module for file and directory operations
import os

# Specify the folder containing the CSV file(s)
# Initialize variable to store total file size
path = "/Users/saiteja/Documents/Assignment"
# Initialize variable to store total file size
totalSize = 0
# Calculate the total size of all CSV files in the directory
for file in os.listdir(path):
    if file.endswith(".csv"):
        totalSize += os.path.getsize(os.path.join(path, file))
# Display the total dataset size in megabytes
print("Total size in MB:", totalSize / (1024 * 1024))
# Count and display the total number of rows in the dataset
print("Total rows: ", df.count())
# Count and display the total number of columns in the dataset
print("Total Columns: ", len(df.columns))

Total size in MB: 13411.494633674622


[Stage 3:=====================================================> (102 + 2) / 105]

Total rows:  44818365
Total Columns:  11


In [7]:
# Count and display the number of NULL (missing) values in each column of the DataFrame
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

[Stage 6:======================================================>(104 + 1) / 105]

+-------+-------+---------+--------+-----------+----------+------------+--------+----------+--------+----------+
|user_id|book_id|review_id|  rating|review_text|date_added|date_updated| read_at|started_at| n_votes|n_comments|
+-------+-------+---------+--------+-----------+----------+------------+--------+----------+--------+----------+
|      0|8053219| 11961490|15331838|   18395970|  25598286|    26570867|32172756|  36200191|33665881|  34417732|
+-------+-------+---------+--------+-----------+----------+------------+--------+----------+--------+----------+



In [8]:
# Remove all rows from the DataFrame that contain any NULL (missing) values
df = df.dropna()

In [10]:
# Parition count
print("Partitions count: ", df.rdd.getNumPartitions())

Partitions count:  105


In [9]:
# Randomly sample 20% of the dataset for faster processing (seed ensures reproducibility)
df = df.sample(0.2, seed=42)
# Persist the sampled DataFrame in memory for faster repeated access
df = df.persist()
# Trigger computation and count total rows in the sampled dataset
df.count()

1237745

In [11]:
# Import PySpark SQL functions and ML libraries for data preprocessing, feature engineering, and pipe
from pyspark.sql.functions import col, length, when, to_timestamp, expr
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

In [12]:
from pyspark.sql.functions import expr

# Convert the columns 'rating', 'n_votes', and 'n_comments' from string to double type using try_cast
df = df.withColumn("rating", expr("try_cast(rating as double)"))
df = df.withColumn("n_votes", expr("try_cast(n_votes as double)"))
df = df.withColumn("n_comments", expr("try_cast(n_comments as double)"))

In [13]:
# Convert 'date_added' column from string to timestamp using specified format
df = df.withColumn(
    "date_added",
    to_timestamp(col("date_added"), "EEE MMM dd HH:mm:ss Z yyyy")
)

# Convert 'date_updated' column from string to timestamp using specified format
df = df.withColumn(
    "date_updated",
    to_timestamp(col("date_updated"), "EEE MMM dd HH:mm:ss Z yyyy")
)

# Convert 'read_at' column from string to timestamp using specified format
df = df.withColumn(
    "read_at",
    to_timestamp(col("read_at"), "EEE MMM dd HH:mm:ss Z yyyy")
)


# Convert 'started_at' column from string to timestamp using specified format
df = df.withColumn(
    "started_at",
    to_timestamp(col("started_at"), "EEE MMM dd HH:mm:ss Z yyyy")
)

In [14]:
# Create a new column 'review_length' that stores the length of each review text
df = df.withColumn("review_length", length(col("review_text")))


# Create a binary column 'has_text' to indicate if review_text is not null (1 = has text, 0 = no text)
df = df.withColumn(
    "has_text",
    when(col("review_text").isNotNull(), 1).otherwise(0)
)

In [15]:
# Create 'engagement_score' by summing number of votes and comments
df = df.withColumn(
    "engagement_score",
    col("n_votes") + col("n_comments")
)

In [16]:
# Convert 'user_id' categorical values into numeric indices for ML models
user_indexer = StringIndexer(
    inputCol="user_id",
    outputCol="user_index",
    handleInvalid="keep" # Keeps unseen or invalid labels instead of failing
)

# Convert 'book_id' categorical values into numeric indices for ML models
book_indexer = StringIndexer(
    inputCol="book_id",
    outputCol="book_index",
    handleInvalid="keep" # Keeps unseen or invalid labels instead of failing
)

In [17]:
# List of selected feature columns used for machine learning model training
feature_cols = [
    "rating",
    "n_votes",
    "n_comments",
    "review_length",
    "engagement_score",
    "user_index",
    "book_index",
    "has_text"
]

In [18]:
# Combine selected feature columns into a single feature vector for ML models
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

In [19]:
# Remove rows where essential columns have missing values
df = df.dropna(subset=["rating", "n_votes", "n_comments"])


In [20]:
# Initialize StandardScaler for feature scaling in PySpark ML pipeline
scaler = StandardScaler(
    inputCol="features", # Input column containing feature vector
    outputCol="scaledFeatures", # Output column for scaled feature vector
    withStd=True,   # Scale features to unit standard deviation
    withMean=False  # Do not center data (keeps sparsity and efficiency)
)

In [21]:
# Create a Machine Learning Pipeline to streamline preprocessing and feature engineering steps
pipeline = Pipeline(stages=[
    user_indexer, # Converts user IDs into numeric indices
    book_indexer,   # Converts book IDs into numeric indices
    assembler, # Combines multiple feature columns into a single feature vector
    scaler  # Standardizes features to improve model performance
])

In [22]:
# Fit the pipeline on the dataset to learn transformations 
model = pipeline.fit(df)
# Apply the fitted pipeline to transform the dataset into a processed format for ML
processed_df = model.transform(df)

26/06/07 21:17:26 WARN DAGScheduler: Broadcasting large task binary with size 20.2 MiB
26/06/07 21:17:35 WARN DAGScheduler: Broadcasting large task binary with size 20.2 MiB
                                                                                

In [23]:
# Display the first 5 rows of the scaled feature vectors without truncation
processed_df.select("scaledFeatures").show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------+
|scaledFeatures                                                                                                                             |
+-------------------------------------------------------------------------------------------------------------------------------------------+
|[0.7354472763096996,0.8059407581089936,2.086999942670629,1.1530797957651064,1.1284303146280816,3.5000523478043846,0.007610358442367365,0.0]|
|[0.7354472763096996,0.0,0.0,0.6658629806530896,0.0,3.7420656614144137,0.009019288757083343,0.0]                                            |
|[0.7354472763096996,0.0,0.0,1.0434560123649026,0.0,3.3286579012856268,4.26948580216963E-5,0.0]                                             |
|[0.5883578210477597,0.0,0.0,0.12586434390393766,0.0,1.1436509035179097,5.443594397766278E-4,0.0]                                           |
|[0.44

26/06/07 21:17:35 WARN DAGScheduler: Broadcasting large task binary with size 17.8 MiB
